# 30 — Interpretability

In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Interpretability helps you debug models, understand failure modes, and build trust. It does not automatically make a model safe, but it gives useful tools: hooks, attention inspection, ablations, probes, and logit lens.

In [ ]:
class HookedMLP(nn.Module):
    def __init__(self, d=8, hidden=16, out=4):
        super().__init__()
        self.fc1 = nn.Linear(d, hidden)
        self.fc2 = nn.Linear(hidden, out)
    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)

model = HookedMLP()
activations = {}
def save_hook(name):
    def hook(module, inp, out):
        activations[name] = out.detach()
    return hook

handle = model.fc1.register_forward_hook(save_hook('fc1'))
x = torch.randn(5,8)
logits = model(x)
print('captured activation shape:', activations['fc1'].shape)
handle.remove()


In [ ]:
def ablate_neuron(model, x, neuron_idx):
    saved = {}
    def hook(module, inp, out):
        y = out.clone()
        y[:, neuron_idx] = 0
        return y
    h = model.fc1.register_forward_hook(hook)
    out = model(x)
    h.remove()
    return out

base = model(x)
ablated = ablate_neuron(model, x, neuron_idx=0)
print('mean logit change after ablating neuron 0:', float((base-ablated).abs().mean()))


In [ ]:
# Probe: can a linear classifier recover a synthetic feature from activations?
X = torch.randn(200,8)
y = (X[:,0] + X[:,1] > 0).long()
with torch.no_grad():
    H = F.relu(model.fc1(X))
probe = nn.Linear(H.size(1), 2)
opt = torch.optim.AdamW(probe.parameters(), lr=0.05)
for step in range(100):
    pred = probe(H)
    loss = F.cross_entropy(pred, y)
    opt.zero_grad(); loss.backward(); opt.step()
acc = (probe(H).argmax(-1) == y).float().mean()
print('probe accuracy:', float(acc))


## Founder notes

Use interpretability for debugging and risk analysis, but do not overclaim. For production trust, combine interpretability with evals, red-team tests, monitoring, human review, and clear UX.